# Wildfire data from remote sensing

This notebook retrieves near-real-time wildfire data and displays it on an interactive map. The user input allows selection of a continent and any date after 2000-01-11 and a day-range between 1 and 5. The day-range defines the number of days before the enddate, that will be displayed.

All observed wildfires in the selected area and time period are then displayed on a map which is opened in your browser. 

Please add your input in the next section:

In [53]:
#user input section 
userdefined_area = 'europe' #any of the continents listed in the list 'continents' below
enddate = '2026-05-21' #enddate of observation period. any date after 2000-01-1 in the string format 'YYYY-MM-DD'
day_range = 1 #day-range to be observed. in range(1,5)

In [54]:
#list of continents that can be selected by user
continents = [
    "Africa",
    "Antarctica",
    "Asia",
    "Europe",
    "North America",
    "South America",
    "Australia",
    "Oceania"
]

## API request
The API https://firms.modaps.eosdis.nasa.gov/api/ is queried for the wildfire observations based on the user input:

- set up API connection
- map user input continent to a bounding box for the API request
- select satellite sensor based on input date
- retrieve data

In [55]:
#setup
import pandas as pd
import requests
import geopandas as gpd
import folium
import numpy as np
import jenkspy
from datetime import datetime
import matplotlib.pyplot as plt
import os
import webbrowser


set up API connection

In [56]:
MAP_KEY = '5eae605403f5deded880b550afef3667' #API key

#count number of API transactions
def get_transaction_count() :
  '''
  Counts transactions used by an API request.

  Returns
  -----------------
  integer
    count of API transactions used by this request
  '''
  count = 0
  try:
    response = requests.get(url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY)
    data = response.json()
    df = pd.Series(data)
    count = df['current_transactions']
  except:
    print ("Error in our call.")
  return count

map user input continent to a bounding box for the API request

In [57]:
#dict to map continent names to bounding boxes
continents_bounding_boxes = {
    "africa": "-20,-37.5,52,37.5",
    "antarctica": "-180,-90,180,-60",
    "asia": "25,-10,180,82",
    "europe": "-31.5,34,66,82",
    "north america": "-170,7,-52,84",
    "south america": "-82,-56,-34,13",
    "australia": "113,-44,154,-10",
    "oceania": "110,-50,180,-10",
    "world": "-180,-90,180,90"
}

#evaluate if the input is a name or a bbox and return bbox
def get_continent_bbox(continent_name):
    '''
    Maps a continent name to a bounding box that can be used for the API request

    Parameters
    -------------
    continent_name: string
        Input name of continent, capitalized or not.
    
    Returns
    ------------
    string
        String with bounding box in format 'lonmin,latmin,lonmax,latmax'   
    '''
    # First check if the input is already a coordinate string
    if isinstance(continent_name, str) and ',' in continent_name:
        parts = continent_name.split(',')
        if len(parts) == 4:
            try:
                # Validate that all parts are numbers
                [float(x) for x in parts]
                return continent_name  # Return as-is if valid coordinates
            except ValueError:
                pass  # Not valid coordinates, proceed to continent check

    # If not coordinates, treat as continent name
    standardized_name = continent_name.strip().lower()
    bbox = continents_bounding_boxes.get(standardized_name)
    if bbox is None:
        print('No Bounding box could be matched to your input.')
    return bbox 

select satellite sensor based on input date and retrieve data

In [58]:
#info dataframe about sensors and their min and max dates

#sensors:
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
daterange_df = pd.read_csv(da_url)
daterange_df['min_date'] = pd.to_datetime(daterange_df['min_date'], format = '%Y-%m-%d')
daterange_df['max_date'] = pd.to_datetime(daterange_df['max_date'], format = '%Y-%m-%d')

display(daterange_df)

#set one sensor
daterange_df.info()

,data_id,min_date,max_date
0,MODIS_NRT,2026-03-01,2026-05-22
1,MODIS_SP,2000-11-01,2026-02-28
2,VIIRS_NOAA20_NRT,2026-04-01,2026-05-21
3,VIIRS_NOAA20_SP,2018-04-01,2026-03-31
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-22
5,VIIRS_SNPP_NRT,2026-04-01,2026-05-22
6,VIIRS_SNPP_SP,2012-01-20,2026-03-31
7,LANDSAT_NRT,2022-06-20,2026-05-21
8,GOES_NRT,2022-08-09,2026-05-22
9,BA_MODIS,2000-11-01,2026-02-01


<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   data_id   11 non-null     str           
 1   min_date  11 non-null     datetime64[us]
 2   max_date  11 non-null     datetime64[us]
dtypes: datetime64[us](2), str(1)
memory usage: 396.0 bytes


In [59]:
#retrieve data 

#parameters for API call
area = get_continent_bbox(userdefined_area) #either "world" or bbox lonmin,latmin,lonmax,latmax, e. g. 0,35,25,70 for europe, "-20,-37.5,52,37.5"  for africa
if enddate is None:
    enddate = datetime.now() #get todays date
else:
    enddate = pd.to_datetime(enddate, format = '%Y-%m-%d')


#set sensor parameter based on enddate
#MODIS_NRT where available
#else: MODIS_SP
#error if neither of them is available
mindate_NRT = daterange_df['min_date'][daterange_df['data_id'] == 'VIIRS_NOAA20_NRT'].values[0]
mindate_SP = daterange_df['min_date'][daterange_df['data_id'] == 'VIIRS_NOAA20_SP'].values[0]
mindate_modis = daterange_df['min_date'][daterange_df['data_id'] == 'MODIS_SP'].values[0]



request_data = True #flag to prevent data request for invalid enddate

#set sensor based on enddate
if enddate > mindate_NRT:
    sensor = 'VIIRS_NOAA20_NRT'
    print('The displayed data is non-processed and not of science quality.')
elif enddate >= mindate_SP:
    sensor = 'VIIRS_NOAA20_SP'
elif enddate >= mindate_modis:
    sensor = 'MODIS_SP'
else:
    print(f"Out of date range, please select an end date after {mindate_modis}")
    request_data = False


print("Current sensor name: ", sensor)

#data request
if request_data:
    area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + f'/{sensor}/{area}/{day_range}/' + enddate.strftime(format = '%Y-%m-%d')
    start_count = get_transaction_count()
    df_area = pd.read_csv(area_url)
    end_count = get_transaction_count()
    print ('We used %i transactions.' % (end_count-start_count))
    display(df_area.head())
    display(df_area.shape)

The displayed data is non-processed and not of science quality.
Current sensor name:  VIIRS_NOAA20_NRT
We used 4 transactions.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,60.99980,60.19313,340.92,0.69,0.75,2026-05-21,37,N20,VIIRS,n,2.0NRT,274.59,10.35,D
1,61.00053,60.19708,348.71,0.69,0.75,2026-05-21,37,N20,VIIRS,n,2.0NRT,277.42,17.90,D
2,61.00554,60.18757,332.93,0.69,0.74,2026-05-21,37,N20,VIIRS,n,2.0NRT,273.25,17.90,D
3,61.00918,60.19455,356.13,0.69,0.74,2026-05-21,37,N20,VIIRS,l,2.0NRT,276.20,47.64,D
4,61.00990,60.19866,356.51,0.69,0.74,2026-05-21,37,N20,VIIRS,l,2.0NRT,279.61,17.90,D


(1572, 14)

## clean the obtained data
- remove low confidence entries
- proper data formats
- convert to GeoPandasDataFrame
- create subset of GeoPandasDataFrame to speed up subsequent processing

In [60]:
#clean df_area:


#remove low confidence entries
if sensor == 'MODIS_SP':
    mask = df_area['confidence'] > 30
else:
    mask = df_area['confidence'] != "l"

print(mask.sum(), "Entries removed due to low confidence")
df_area = df_area[mask]

#format datetime
df_area["acq_datetime"] = pd.to_datetime(df_area['acq_date'] + df_area['acq_time'].astype(str).str.zfill(4),  # Zero-pad to 4 digits (e.g., '626' -> '0626')
    format='%Y-%m-%d%H%M', errors='coerce'
)

df_area = df_area.drop(columns = ["acq_date", "acq_time"])

1497 Entries removed due to low confidence


In [61]:
#convert to GeoPandas and create subset of dataframe with only the relevant columns to speed up subsequent processing

area_gpd = gpd.GeoDataFrame(
    df_area, geometry=gpd.points_from_xy(df_area["longitude"], df_area["latitude"], crs = 4326)
)
area_gpd_sub = area_gpd[['frp', 'acq_datetime', 'geometry']].copy()
area_gpd_sub['acq_datetime'] = area_gpd_sub['acq_datetime'].astype(str)



## intersect fire data with countries

the file countries_simple.gpkg is obtained from the countries.ipynb notebook
- spatially join fire data to country polygons
- count number of fires per country
- merge the result with the country gdf

In [62]:
#load countries geopandas and spatially join with fire
countries_simple = gpd.read_file('data/countries_simple.gpkg').to_crs(4326)
fire_countries_merged = area_gpd_sub.sjoin(countries_simple, how = 'left', predicate = 'within')  #fires within countries
fire_countries_merged.head()

,frp,acq_datetime,geometry,index_right,CountryName
0,10.35,2026-05-21 00:37:00,POINT (60.19313 60.9998),140.0,Russia
1,17.90,2026-05-21 00:37:00,POINT (60.19708 61.00053),140.0,Russia
2,17.90,2026-05-21 00:37:00,POINT (60.18757 61.00554),140.0,Russia
5,1.27,2026-05-21 00:39:00,POINT (38.84198 55.06178),140.0,Russia
6,1.80,2026-05-21 00:39:00,POINT (37.76862 55.55544),140.0,Russia


In [63]:
#count number of fires that could not be matched to a country
print(len(area_gpd_sub)- fire_countries_merged.groupby('CountryName')['frp'].count().sum(), "fires were not matched to a country") #some fires are not matched to a country. probably due to the polygon simplification

16 fires were not matched to a country


In [64]:
#count number of fires by country

#group fire occurences by country and count
fire_countries_count = pd.DataFrame(fire_countries_merged.groupby('CountryName')['frp'].count()) #take any random column for the count, here 'frp'
fire_countries_count = fire_countries_count.rename(columns= {'frp': 'firecount'}) #rename column to count

#merge with countries_simple so it is a gpd and calculate fire density
fire_countries_count = pd.merge(countries_simple, fire_countries_count, how = 'left', on = 'CountryName')
fire_countries_count['area_km2'] = (fire_countries_count.geometry.to_crs(8857).area / 10**6).round(1) #mind the crs conversion to equal area!
fire_countries_count['fires/100\'000km2'] = (fire_countries_count['firecount']/fire_countries_count['area_km2']*100000).round(3)
display(fire_countries_count.head())

,CountryName,geometry,firecount,area_km2,fires/100'000km2
0,Afghanistan,"MULTIPOLYGON (((74.87679 37.2749, 74.88489 37....",NaN,641718.1,NaN
1,United Kingdom,"MULTIPOLYGON (((33.013 34.64377, 33.00785 34.5...",2.0,251619.5,0.795
2,Albania,"MULTIPOLYGON (((20.07889 42.55579, 20.16519 42...",1.0,28709.7,3.483
3,Algeria,"MULTIPOLYGON (((8.64195 36.94096, 8.65373 36.9...",76.0,2306853.2,3.295
4,United States,"MULTIPOLYGON (((-168.15542 -14.51823, -168.165...",NaN,9495212.8,NaN


## visualise

- add a rectangle showing the bounding box that was selected based on the user input
- cluster the fire observations
- add choropleth per country for fire density (including hover-over feature)
- add title and legend to the output html file
- open the html file in the webbrowser

In [65]:
from folium.plugins import MarkerCluster


#define mapcenter based on selected continent
lonmin,latmin,lonmax,latmax = map(float, area.split(','))
mapcenter = [(latmin+latmax)/2, (lonmin+lonmax)/2]

# Initialize the basemap
m6 = folium.Map(tiles="CartoDB Positron", zoom_start = 3, location = mapcenter)

# Add the bboxrectangle to the map
folium.Rectangle(
    bounds=[[latmin, lonmin], [latmax, lonmax]],  # [[south, west], [north, east]]
    color='blue',
    fill=False,
    weight=2,
    popup = "Selected Area"
).add_to(m6)


#############################################
#add clusters
marker_cluster = MarkerCluster(name="Scaled Wildfire Clusters", cmap = "YlOrRd").add_to(m6)

for idx, row in area_gpd_sub.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x
    frp = row["frp"]
    tooltip_text = f"Fire Reactive Power: {frp}"

    # Mathematical Scaling Logic:
    # Base size of 14px, plus an increase based on the square root of the frp
    icon_size = 12 + (np.sqrt(frp) * 30)  # <- THIS IS NEW

    # Injecting custom CSS to draw a perfect circle with our dynamic size (NEW)
    icon_html = f"""

        <div style="
            font-size: {icon_size}px;
            color: #FF4500;
            background: #FF8C00;
            border-radius: 50%;
            width: {icon_size}px;
            height: {icon_size}px;
            display: flex;
            align-items: center;
            justify-content: center;
            border: 1px solid #1f77b4;">
            <i class="fa fa-fire"></i>
        </div>"""


    # Apply the custom HTML using DivIcon
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(  # <- THIS IS NEW
            html=icon_html,
            icon_size=(icon_size, icon_size),
            icon_anchor=(icon_size / 2, icon_size / 2),  # Centers the icon perfectly
        ),
        tooltip=tooltip_text,
    ).add_to(marker_cluster)


######################################

#fire density choropleth


#check if there is more than one country with fires in it, otherwise the choropleth does not work
if fire_countries_count['fires/100\'000km2'].count() > 1:    
    #add fire density choropleth
    vmin = fire_countries_count["fires/100'000km2"].quantile(0.05)
    vmax = fire_countries_count["fires/100'000km2"].quantile(0.8)

    # Clone the dataframe and clip values
    #otherwise the vmin and vmax don't work and the colorbar is affected by outliers
    data_clipped = fire_countries_count.copy()
    data_clipped["fires/100'000km2"] = data_clipped["fires/100'000km2"].clip(lower=vmin, upper=vmax)

    folium.Choropleth(
    geo_data=fire_countries_count,
    name="Fire Density (Fires/100'000km2)",
    data=data_clipped,
    columns=["CountryName", "fires/100'000km2"],
    key_on="feature.properties.CountryName",
    fill_color="YlOrRd",
    fill_opacity=0.6,
    line_opacity=0.2,
    legend_name="Fires per 100'000km2",
    vmin = vmin,
    vmax = vmax, 
    nan_fill_color= 'white', 
    bins = 10,
    use_jenks= fire_countries_count['fires/100\'000km2'].count() > 10 #only use jenks if more classes than bins
    ).add_to(m6)

#######################################
#add invisible density choropleth hover over feature
folium.GeoJson(
    fire_countries_count,
    name="Interactive Tooltips",
    # Make the polygons completely transparent so they do not hide the choropleth colors
    style_function=lambda x: {"fillColor": "#ffffff00", "color": "#ffffff00"},
    tooltip=folium.GeoJsonTooltip(
        fields=["CountryName", "fires/100'000km2"],
        aliases=["Country:", "Fire Density:"],
        localize=True,
    ),
).add_to(m6)

##################################3
# Add the interactive layer control menu
folium.LayerControl().add_to(m6)



#add title to map and save it as html file

# Add a title using folium.Element

#define startdate for the title
startdate = enddate - pd.to_timedelta(day_range-1, 'D')

title_html = f'''
             <h3 align="center" style="font-size:16px"><b>Wildfires in selected area from {startdate.strftime(format = '%Y-%m-%d')} to {enddate.strftime(format = '%Y-%m-%d')}</b></h3>
             '''

m6.get_root().html.add_child(folium.Element(title_html))

#create legend for the blue rectangle bbox
# Create HTML for the legend
legend_html = '''
<div style="position: fixed; 
     bottom: 10px; right: 10px; width: 150px; height: 70px; 
     background-color: white; border:2px solid grey; z-index:9999; 
     font-size:16px; padding: 5px">
     
    <p style="margin: 0 0 5px 0;"><b>Legend</b></p>
    
    <p style="margin: 5px 0;">
        <svg width="20" height="20" style="vertical-align: middle;">
            <rect x="2" y="2" width="16" height="16" 
                  fill="none" stroke="blue" stroke-width="2"/>
        </svg>
        Selected Area
    </p>
</div>
'''
# Add the legend to the map
m6.get_root().html.add_child(folium.Element(legend_html))



file_path = 'docs/index.html'
m6.save(file_path)

#define boundaries of displayed map
m6.fit_bounds([(latmin, lonmin),(latmax, lonmax)])  # [(latmin, lonmin), (latmax, lonmax)]

#m6 #displays map in the notebook, without title and legend

webbrowser.open_new_tab(f'file://{os.path.realpath(file_path)}') #opens map in the browser, including title and legend

print("The map opened in your browser")

The map opened in your browser
